# P1.6 — Pretrain GPT-2 124M on an A100, and measure its quality

A **fresh** run with the GPT-2 reference recipe (`scripts/train_a100.py`: 524,288-token
batches, 2.5B tokens ≈ 4,768 steps, ~6 h). Quality is measured by **log-likelihood**:
validation loss (perplexity, bits/token) every 250 steps and **HellaSwag** every 500.

**Everything durable lives on Google Drive** — training data (~5 GB) and checkpoints
(2 kept, ~1.5 GB each). If Colab disconnects, re-run cells **2 → 4**, then **7**: the
run **resumes from its latest checkpoint**, not from step 0.

**Before running:** write your predictions (final val loss, HellaSwag `acc_norm`, MFU)
in `docs/benchmarks/training.md` — step **d** of P1.6.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2 · Drive and paths
from google.colab import drive
drive.mount('/content/drive')
DRIVE      = '/content/drive/MyDrive/gpt2-from-scratch'
DRIVE_DATA = f'{DRIVE}/edu_fineweb10B'      # durable copy of the shards
DATA       = '/content/edu_fineweb10B'      # local copy the run reads (faster than Drive)
RUN        = f'{DRIVE}/runs_a100'           # checkpoints + log.txt
EVAL       = f'{DRIVE}/eval'
!mkdir -p "$DRIVE" "$EVAL"

In [ ]:
# 3 · code: clone once, pull on later sessions
import os
if not os.path.exists('/content/gpt2-from-scratch'):
    !git clone -q https://github.com/shuzhancnjx/gpt2-from-scratch.git /content/gpt2-from-scratch
%cd /content/gpt2-from-scratch
!git pull -q
!pip install -q -e ".[dev]" datasets

In [ ]:
# 4 · data: 1 val + 250 train shards (2.5B tokens). Built once (~30–60 min, streamed,
#     never the full 10B), kept on Drive, copied to local disk each session.
import glob
n_drive = len(glob.glob(f'{DRIVE_DATA}/edufineweb_*'))
if n_drive >= 251:
    print(f'{n_drive} shards on Drive -> copying to local disk')
    !mkdir -p "$DATA" && cp -n "$DRIVE_DATA"/edufineweb_* "$DATA"/
else:
    !python scripts/prepare_data.py --out-dir "$DATA" --max-shards 251
    !mkdir -p "$DRIVE_DATA" && cp -n "$DATA"/edufineweb_* "$DRIVE_DATA"/
print(len(glob.glob(f'{DATA}/edufineweb_*')), 'shards ready locally')

## 5 · Baseline on this GPU — OpenAI's GPT-2 124M

The target, measured with the same code and the same validation window as your run.
(Your old checkpoint's baseline was measured on the Mac: `docs/benchmarks/training.md`.)

In [ ]:
!python scripts/eval_checkpoint.py --hf gpt2 --data-root "$DATA" --out "$EVAL/openai_gpt2_a100.json"

## 6 · Smoke test — and prove that resume works

Two short runs into a throwaway directory. The first stops after 12 steps
(checkpoint at step 10 and at its last step, 11). The second asks for 20 steps with
`resume_from='latest'`: it must print **`resumed from … at step 12`**, not start at 0.
HellaSwag is limited to 200 examples here so the test is quick.

In [ ]:
import dataclasses, importlib.util, shutil
from gpt2.training import train
spec = importlib.util.spec_from_file_location('train_a100', 'scripts/train_a100.py')
train_a100 = importlib.util.module_from_spec(spec); spec.loader.exec_module(train_a100)

SMOKE = '/content/runs_smoke'
shutil.rmtree(SMOKE, ignore_errors=True)
base = dataclasses.replace(train_a100.CONFIG, data_root=DATA, run_dir=SMOKE, eval_every=10,
                           hellaswag_every=10, hellaswag_limit=200, lr_schedule_steps=20)
train(dataclasses.replace(base, max_steps=12))                       # first run: steps 0..11
train(dataclasses.replace(base, max_steps=20, resume_from='latest'))  # must resume at 12

## 7 · The full run (~6 h)

Fresh on the first session; on any later session `resume_from='latest'` picks up the
newest checkpoint in `RUN`. Re-run cells 2 → 4, then this one.

In [ ]:
cfg = dataclasses.replace(train_a100.CONFIG, data_root=DATA, run_dir=RUN, resume_from='latest',
                          keep_last_n_checkpoints=2, hellaswag_every=500)
model = train(cfg)

## 8 · Curves — run any time, even mid-training

In [ ]:
import json, math, matplotlib.pyplot as plt
from gpt2.log import read_log
log = read_log(f'{RUN}/log.txt')
TOK_PER_STEP = 524288
ref = json.load(open(f'{EVAL}/openai_gpt2_a100.json')) if os.path.exists(f'{EVAL}/openai_gpt2_a100.json') else None

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for split, style in (('train', dict(alpha=.3, lw=.6)), ('val', dict(marker='o', ms=3))):
    s = log[split]
    ax[0].plot([x * TOK_PER_STEP / 1e9 for x, _ in s], [v for _, v in s], label=split, **style)
if ref: ax[0].axhline(ref['val']['nll'], ls='--', c='k', label='OpenAI GPT-2 (val)')
ax[0].set(xlabel='tokens (B)', ylabel='loss (nats/token)', title='loss', ylim=(2.8, 6)); ax[0].legend()

h = log.get('hella', [])
ax[1].plot([x * TOK_PER_STEP / 1e9 for x, _ in h], [v for _, v in h], marker='o')
ax[1].axhline(0.25, ls=':', c='gray', label='random (25%)')
if ref: ax[1].axhline(ref['hellaswag']['acc_norm'], ls='--', c='k', label='OpenAI GPT-2')
ax[1].set(xlabel='tokens (B)', ylabel='acc_norm', title='HellaSwag'); ax[1].legend()

# MFU: model FLOPs per token ≈ 6N (forward 2N + backward 4N) + attention 12·L·d·T
N, L, d, T, PEAK = 124e6, 12, 768, 1024, 312e12
flops_per_token = 6 * N + 12 * L * d * T
t = log.get('tok_s', [])[5:]                 # skip compile/warm-up steps
ax[2].plot([x for x, _ in t], [100 * v * flops_per_token / PEAK for _, v in t], lw=.6)
ax[2].set(xlabel='step', ylabel='MFU (%)', title='model FLOPs utilisation')
plt.tight_layout(); plt.show()
if log['val']:
    s, v = log['val'][-1]
    print(f'latest val @ step {s}: nll {v:.4f}  perplexity {math.exp(v):.1f}')
if h: print(f'latest HellaSwag @ step {h[-1][0]}: acc_norm {h[-1][1]:.4f}')

## 9 · Final evaluation — same script, same window as every baseline

In [ ]:
latest = sorted(glob.glob(f'{RUN}/model_*.pt'))[-1]
!python scripts/eval_checkpoint.py --checkpoint "$latest" --data-root "$DATA" --out "$EVAL/final_a100.json"

## 10 · Export for the chat UI

Weights + config only (no optimizer): ~0.5 GB. Download it, then on the Mac:
`python scripts/chat_server.py --checkpoint <path>`.

In [ ]:
import torch
ck = torch.load(latest, map_location='cpu', weights_only=False)
out = f'{DRIVE}/gpt2_124M_a100_step{ck["step"]:06d}.pt'
torch.save({'config': ck['config'], 'model': ck['model'], 'step': ck['step']}, out)
print('saved', out)

## 11 · Your analysis (judged — P1.6 h)

In `docs/benchmarks/training.md`, using the curves and the final table:

1. **Prediction vs result** — final val loss, HellaSwag `acc_norm` and MFU against what
   you predicted in step d. Where were you wrong, and why?
2. **Two kinds of quality** — val loss improves smoothly; does HellaSwag? When does it
   leave the 25% random line, and why might it lag the loss?
3. **Versus OpenAI GPT-2** — compare val loss *and* HellaSwag. Why is the val-loss
   comparison only rough (hint: what was each model trained on?), while HellaSwag is fair?
4. **Loss by position** (from `eval_checkpoint.py`) — how much does context help the
   new model compared with the old checkpoint?
5. **MFU** — explain the measured value with the roofline: what keeps it below 100%?
6. **Before / after** — the same prompts in the chat UI with the old and new checkpoints.